# Set Up

In [1]:
# Setup environment and imports
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Optional dependencies
try:
    import openpyxl  # noqa: F401
except ImportError:
    openpyxl = None

try:
    import pyarrow  # noqa: F401
except ImportError:
    pyarrow = None

plt.style.use('seaborn-v0_8')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 80)

RAW_DATA_DIR = Path('/Users/matthewpitlock/Development/FantasyBaseballAnalytics/data/raw')
RAW_DATA_DIR.exists()


True

# Table Ingestion

In [20]:
# Define the raw source files
source_files = {
    # player contract data
    'sda_tracker_excel': RAW_DATA_DIR / 'SDA - Major League Contract Tracker 080126.xlsx',

    # fantrax data
    'fantrax_all_hitter_stats': RAW_DATA_DIR / 'Fantrax-Players-allhitterstats_080126.csv',
    'fantrax_all_pitcher_stats': RAW_DATA_DIR / 'Fantrax-Players-allpitcherstats_080126.csv',
    'fantrax_sda_players': RAW_DATA_DIR / 'Fantrax-Players-Second Deck Association.csv',

    # player rankings
    'keith_prospects': RAW_DATA_DIR / 'Keith_Top_Prospects.csv',
    'harryknowsball': RAW_DATA_DIR / 'harryknowsball_daynasty_ranking.csv',
}

available_sources = {
    key: path.name for key, path in source_files.items() if path.exists()
}

available_sources


{'sda_tracker_excel': 'SDA - Major League Contract Tracker 080126.xlsx',
 'fantrax_all_hitter_stats': 'Fantrax-Players-allhitterstats_080126.csv',
 'fantrax_all_pitcher_stats': 'Fantrax-Players-allpitcherstats_080126.csv',
 'fantrax_sda_players': 'Fantrax-Players-Second Deck Association.csv',
 'keith_prospects': 'Keith_Top_Prospects.csv',
 'harryknowsball': 'harryknowsball_daynasty_ranking.csv'}

In [11]:
# Load and inspect each source file
from pprint import pprint

loaded = {}

for name, path in source_files.items():
    if not path.exists():
        print(f'{name}: missing')
        continue
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    elif path.suffix.lower() in {'.xlsx', '.xls'}:
        xl = pd.ExcelFile(path)
        print(f'{name}: workbook sheets -> {xl.sheet_names}')
        # Use the first sheet for an initial preview.
        df = pd.read_excel(path, sheet_name=xl.sheet_names[0])
    else:
        continue
    loaded[name] = df
    print(f'\n=== {name} ===')
    print('shape:', df.shape)
    print('columns:', list(df.columns)[:15])
    print(df.head(2).to_string(index=False))
    print('---')

loaded.keys()


sda_tracker_excel: workbook sheets -> ['Intro Page', 'League History', 'Example Team Template', 'Blank Salary Tracker (July 14, ', 'Salary Cap Tracking', 'Estimated Free Agent Class', 'Boe', 'Boe - Sorted', 'Maloun', 'Hoffman', 'Pecora', 'C. Pelton', 'W. Pelton', 'Pitlock', 'Riggen', 'A. Rolain', 'M. Rolain']

=== sda_tracker_excel ===
shape: (1, 1)
columns: ['Second Deck Association Contracts Tracking Sheet (*This Tracking sheet is currently a work in progress)']
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

dict_keys(['sda_tracker_excel', 'fantrax_all_stats', 'fantrax_sda_players', 'keith_prospects', 'harryknowsball'])

In [12]:
# Create a normalization plan for downstream use

normalization_plan = {
    'players': {
        'purpose': 'One row per normalized player identity',
        'keys': ['player_id', 'player_name', 'mlb_team', 'position'],
        'notes': 'Normalize player names and reconcile Fantrax, prospect, and ranking sources into a canonical player table.'
    },
    'player_aliases': {
        'purpose': 'Store alternate names and source-specific IDs',
        'keys': ['player_id', 'source_system', 'source_player_id', 'source_name'],
        'notes': 'Use this to map Fantrax IDs, prospect list names, and contract tracker names to the same player.'
    },
    'player_ratings': {
        'purpose': 'Store third-party rankings and projections',
        'keys': ['player_id', 'source_system', 'rank', 'value', 'date'],
        'notes': 'Keep rankings separate from core player facts so they can evolve over time.'
    },
    'roster_status': {
        'purpose': 'Track availability and current team ownership',
        'keys': ['player_id', 'league_year', 'roster_status', 'team_name'],
        'notes': 'This can combine Fantrax availability and SDA contract-tracker ownership data.'
    },
    'contracts': {
        'purpose': 'Store contract terms and ownership history',
        'keys': ['player_id', 'team_name', 'contract_years', 'move_type', 'effective_date'],
        'notes': 'Use this for long-term roster planning and salary-cap analysis.'
    }
}

pprint(normalization_plan)


{'contracts': {'keys': ['player_id',
                        'team_name',
                        'contract_years',
                        'move_type',
                        'effective_date'],
               'notes': 'Use this for long-term roster planning and salary-cap '
                        'analysis.',
               'purpose': 'Store contract terms and ownership history'},
 'player_aliases': {'keys': ['player_id',
                             'source_system',
                             'source_player_id',
                             'source_name'],
                    'notes': 'Use this to map Fantrax IDs, prospect list '
                             'names, and contract tracker names to the same '
                             'player.',
                    'purpose': 'Store alternate names and source-specific IDs'},
 'player_ratings': {'keys': ['player_id',
                             'source_system',
                             'rank',
                             'va

<!-- # Multi Year Contract Player Summary -->

In [15]:
#All team multi year contracts 
from IPython.display import display, HTML

# Use the workbook path from the source mapping defined earlier.
excel_path = source_files.get('sda_tracker_excel')

if excel_path is None:
    raise ValueError('Missing sda_tracker_excel in source_files')

excel_path = Path(excel_path)
if not excel_path.exists():
    raise FileNotFoundError(f'Missing workbook: {excel_path}')

print(f'Using workbook: {excel_path}')

def canonicalize_header(value):
    if pd.isna(value):
        return ''
    cleaned = re.sub(r'[\.\*]', '', str(value).strip())
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def parse_owner_tab(sheet_name):
    raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    header_row_idx = None
    for idx, row in raw.iterrows():
        values = [canonicalize_header(v).lower() for v in row.tolist()]
        if any('player name' in v for v in values) and any('free agent year' in v for v in values):
            header_row_idx = idx
            break

    if header_row_idx is None:
        return None

    header_row = [canonicalize_header(value) for value in raw.iloc[header_row_idx].tolist()]
    tab_df = raw.iloc[header_row_idx + 1:].copy()
    tab_df.columns = header_row

    column_lookup = {canonicalize_header(col).casefold(): col for col in tab_df.columns}
    rename_map = {}
    for target, aliases in {
        'Player Name': ['player name'],
        'Pos': ['pos'],
        'Free Agent Year': ['free agent year'],
        'Move Type': ['move type'],
        'Player Contract Years': ['player contract years'],
    }.items():
        for alias in aliases:
            if alias in column_lookup:
                rename_map[column_lookup[alias]] = target
                break

    if rename_map:
        tab_df = tab_df.rename(columns=rename_map)

    selected_columns = ['Player Name', 'Pos', 'Free Agent Year', 'Move Type', 'Player Contract Years']
    selected_columns = [c for c in selected_columns if c in tab_df.columns]
    tab_df = tab_df[selected_columns].copy()

    team_owner = sheet_name
    for _, row in raw.iterrows():
        for col_idx, value in enumerate(row.tolist()):
            text = canonicalize_header(value).lower()
            if text.startswith('team owner') and col_idx + 1 < len(row):
                candidate_owner = canonicalize_header(row.iloc[col_idx + 1])
                if candidate_owner:
                    team_owner = candidate_owner
                    break
        if team_owner != sheet_name:
            break

    tab_df['Player Name'] = tab_df['Player Name'].fillna('').astype(str).str.strip()
    tab_df['Move Type'] = tab_df['Move Type'].fillna('').astype(str).str.strip()
    tab_df['Player Contract Years'] = pd.to_numeric(tab_df['Player Contract Years'], errors='coerce')

    mask = (
        tab_df['Player Name'].ne('')
        & tab_df['Player Contract Years'].gt(1)
        & tab_df['Move Type'].ne('DFA')
        & tab_df['Move Type'].ne('')
    )
    tab_df = tab_df.loc[mask].copy()
    tab_df['team_owner'] = team_owner
    return tab_df


owner_sheet_names = [
    s for s in pd.ExcelFile(excel_path).sheet_names
    if s not in {
        'Intro Page',
        'League History',
        'Example Team Template',
        'Blank Salary Tracker (July 14,',
        'Salary Cap Tracking',
        'Estimated Free Agent Class',
        'Boe - Sorted',
    }
]

owner_tab_frames = []
for sheet_name in owner_sheet_names:
    parsed = parse_owner_tab(sheet_name)
    if parsed is not None and not parsed.empty:
        owner_tab_frames.append(parsed)

owner_tab_df = pd.concat(owner_tab_frames, ignore_index=True)
owner_tab_df = owner_tab_df[['team_owner', 'Player Name', 'Pos', 'Free Agent Year', 'Move Type', 'Player Contract Years']]

print(f'Loaded {owner_tab_df.shape[0]:,} filtered rows across {owner_tab_df["team_owner"].nunique()} team owner tabs')
print(owner_tab_df.head(20).to_string(index=False))

html_table = owner_tab_df.to_html(index=False)
scrollable_html = f"<div style='max-height:500px; overflow:auto; border:1px solid #ddd; padding:8px;'>{html_table}</div>"
display(HTML(scrollable_html))

Using workbook: /Users/matthewpitlock/Development/FantasyBaseballAnalytics/data/raw/SDA - Major League Contract Tracker 080126.xlsx
Loaded 170 filtered rows across 10 team owner tabs
team_owner       Player Name Pos Free Agent Year Move Type  Player Contract Years
       Boe     Michael Busch  1B            2030     Draft                    4.0
       Boe    Kazuma Okamoto  1B            2028  Promoted                    2.0
       Boe     JJ Wetherholt  2B            2033     Draft                    7.0
       Boe      Bo Bischette  3B            2031     Draft                    5.0
       Boe    Samuel Basallo   C            2033     Draft                    7.0
       Boe     Roman Anthony  OF            2033     Draft                    7.0
       Boe    Wyatt Langford  OF            2033     Draft                    7.0
       Boe    Addison Barger  OF            2028     Draft                    2.0
       Boe       Bobby Abreu  RP            2028     Draft                    2

team_owner,Player Name,Pos,Free Agent Year,Move Type,Player Contract Years
Boe,Michael Busch,1B,2030,Draft,4.0
Boe,Kazuma Okamoto,1B,2028,Promoted,2.0
Boe,JJ Wetherholt,2B,2033,Draft,7.0
Boe,Bo Bischette,3B,2031,Draft,5.0
Boe,Samuel Basallo,C,2033,Draft,7.0
Boe,Roman Anthony,OF,2033,Draft,7.0
Boe,Wyatt Langford,OF,2033,Draft,7.0
Boe,Addison Barger,OF,2028,Draft,2.0
Boe,Bobby Abreu,RP,2028,Draft,2.0
Boe,Nolan McLean,SP,2033,Draft,7.0


In [ ]:
team_summary = (
    owner_tab_df.groupby('team_owner')
    .agg(
        player_count=('Player Name', 'count'),
        max_free_agent_year=('Free Agent Year', lambda s: pd.to_numeric(s, errors='coerce').max()),
        max_player_contract_years=('Player Contract Years', 'max'),
        sum_player_contract_years=('Player Contract Years', 'sum'),
    )
    .reset_index()
)

print('team_owners:', owner_tab_df['team_owner'].nunique())
print('max free agent year:', int(team_summary['max_free_agent_year'].max()))
print('max player contract years:', float(team_summary['max_player_contract_years'].max()))
print('max sum(player contract years) by team:', float(team_summary['sum_player_contract_years'].max()))
print('max count(player name) for a team:', int(team_summary['player_count'].max()))

team_summary.sort_values('sum_player_contract_years', ascending=False).head(10)

# All Fantrax Player Status

In [17]:
# Load the Fantrax SDA players export from the source mapping defined in cell 4
fantrax_sda_players_path = source_files['fantrax_sda_players']

if fantrax_sda_players_path.exists():
    fantrax_sda_players_df = pd.read_csv(fantrax_sda_players_path)
    print(f'Loaded {fantrax_sda_players_df.shape[0]:,} rows and {fantrax_sda_players_df.shape[1]} columns')
    print('Columns:', list(fantrax_sda_players_df.columns)[:20])
    print('\nPreview:')
    display(fantrax_sda_players_df.head(5))
else:
    fantrax_sda_players_df = None
    print(f'Missing file: {fantrax_sda_players_path}')

# Join Fantrax player status to the roster owner labels from the workbook parser
if 'fantrax_sda_players_df' in globals() and fantrax_sda_players_df is not None:
    def normalize_name(value):
        if pd.isna(value):
            return ''
        text = str(value).strip().lower()
        text = re.sub(r'[^a-z0-9]+', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    owner_tab_df['name_key'] = owner_tab_df['Player Name'].map(normalize_name)
    fantrax_sda_players_df['name_key'] = fantrax_sda_players_df['Player'].map(normalize_name)

    joined_status_owner = (
        fantrax_sda_players_df.merge(
            owner_tab_df[['name_key', 'team_owner']].drop_duplicates(),
            on='name_key',
            how='left'
        )
    )

    # Build an estimated mapping from status -> owner using the most common owner among joined players.
    status_owner_counts = (
        joined_status_owner.dropna(subset=['team_owner'])
        .groupby(['Status', 'team_owner'])
        .size()
        .reset_index(name='count')
        .sort_values(['Status', 'count'], ascending=[True, False])
    )
    status_owner_mapping = (
        status_owner_counts.drop_duplicates('Status')
        .rename(columns={'team_owner': 'owner_estimate'})
        [['Status', 'owner_estimate', 'count']]
    )

    # Force free-agent-like statuses to FA.
    fa_mask = status_owner_mapping['Status'].astype(str).str.upper().str.contains('FA|FREE', na=False)
    status_owner_mapping.loc[fa_mask, 'owner_estimate'] = 'FA'

    # Apply the estimated mapping to rows that are still missing an owner label.
    status_lookup = status_owner_mapping.set_index('Status')['owner_estimate'].to_dict()
    joined_status_owner['owner_estimate'] = joined_status_owner['team_owner']
    missing_owner_mask = joined_status_owner['owner_estimate'].isna()
    joined_status_owner.loc[missing_owner_mask, 'owner_estimate'] = joined_status_owner.loc[missing_owner_mask, 'Status'].map(status_lookup)
    joined_status_owner.loc[
        joined_status_owner['owner_estimate'].isna() & joined_status_owner['Status'].astype(str).str.upper().str.contains('FA|FREE', na=False),
        'owner_estimate'
    ] = 'FA'

    print('\nEstimated status-to-owner mapping:')
    display(status_owner_mapping.sort_values('Status'))

    print('\nStatus-to-owner mapping preview:')
    display(joined_status_owner[['Player', 'Status', 'team_owner', 'owner_estimate']].head(20))

    print('\nEstimated owner counts by status:')
    display(joined_status_owner.groupby(['Status', 'owner_estimate']).size().reset_index(name='count').sort_values(['Status', 'count'], ascending=[True, False]).head(30))


Loaded 7,103 rows and 11 columns
Columns: ['ID', 'Player', 'Team', 'Position', 'RkOv', 'Status', 'Age', 'Opponent', 'Score', 'Ros', '+/-']

Preview:


,ID,Player,Team,Position,RkOv,Status,Age,Opponent,Score,Ros,+/-
0,*05rat*,James Wood,WSH,OF,1,DDDS,23,@ATL Sat 7:15PM,100.00,100%,0%
1,*05u6q*,Cam Schlittler,NYY,SP,2,AOP,25,@CHC Sat 7:15PM,100.00,99%,0%
2,*02yc4*,Shohei Ohtani,LAD,"UT,SP",3,Gloggs,32,BOS Sat 9:10PM,99.70,100%,0%
3,*061yu*,Jacob Misiorowski,MIL,SP,4,BPP,24,@LAA Sat 9:38PM,99.43,100%,0%
4,*04qk8*,CJ Abrams,WSH,SS,5,AOP,25,@ATL Sat 7:15PM,99.22,100%,0%



Estimated status-to-owner mapping:


,Status,owner_estimate,count
1,AOP,Pecora,17
2,BPP,Boe,9
3,CHAS,C. Pelton,13
5,DDDS,Hoffman,12
6,FA,FA,2
12,Gloggs,M. Rolain,20
13,IL 10,Riggen,18
14,PitlockM,Pitlock,14
15,WPP,Maloun,20
16,⭐️✨🌟,A. Rolain,18



Status-to-owner mapping preview:


,Player,Status,team_owner,owner_estimate
0,James Wood,DDDS,Hoffman,Hoffman
1,Cam Schlittler,AOP,Pecora,Pecora
2,Shohei Ohtani,Gloggs,M. Rolain,M. Rolain
3,Jacob Misiorowski,BPP,Boe,Boe
4,CJ Abrams,AOP,Pecora,Pecora
5,Pete Crow-Armstrong,IL 10,Riggen,Riggen
6,Chris Sale,Gloggs,M. Rolain,M. Rolain
7,Yoshinobu Yamamoto,AOP,Pecora,Pecora
8,Mason Miller,Gloggs,M. Rolain,M. Rolain
9,Cristopher Sanchez,CHAS,C. Pelton,C. Pelton



Estimated owner counts by status:


,Status,owner_estimate,count
1,AOP,Pecora,37
0,AOP,Maloun,1
2,BPP,Boe,44
3,CHAS,C. Pelton,46
4,CHAS,Pecora,1
5,DDDS,Hoffman,46
9,FA,FA,6645
6,FA,A. Rolain,2
7,FA,Boe,2
10,FA,Hoffman,2


# Organization, FA and Minor League Build Out

In [19]:
# Build the requested roster-style dataframe
def normalize_name(value):
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Prepare roster source
roster_df = owner_tab_df[['team_owner', 'Player Name', 'Pos', 'Player Contract Years']].copy()
roster_df = roster_df.rename(columns={
    'team_owner': 'Team Owner',
    'Player Name': 'Player',
    'Pos': 'Position',
    'Player Contract Years': 'Player Contract Years',
})
roster_df['name_key'] = roster_df['Player'].map(normalize_name)
roster_df['Player Contract Years'] = pd.to_numeric(roster_df['Player Contract Years'], errors='coerce').fillna(1).astype(int)

# Prepare Fantrax stats source from whichever file is available
if 'fantrax_all_stats_df' in globals() and fantrax_all_stats_df is not None:
    stats_df = fantrax_all_stats_df.copy()
elif 'fantrax_all_hitter_stats' in source_files and source_files['fantrax_all_hitter_stats'].exists():
    stats_df = pd.read_csv(source_files['fantrax_all_hitter_stats'])
    if 'fantrax_all_pitcher_stats' in source_files and source_files['fantrax_all_pitcher_stats'].exists():
        pitcher_stats_df = pd.read_csv(source_files['fantrax_all_pitcher_stats'])
        stats_df = pd.concat([stats_df, pitcher_stats_df], ignore_index=True)
else:
    stats_df = pd.DataFrame()

if not stats_df.empty and 'Player' in stats_df.columns:
    stats_df['name_key'] = stats_df['Player'].map(normalize_name)
else:
    stats_df = pd.DataFrame(columns=['name_key', 'Team', 'RkOv', 'Age'])

# Prepare status source from the earlier join
status_df = joined_status_owner[['Player', 'Status', 'owner_estimate']].copy()
status_df['name_key'] = status_df['Player'].map(normalize_name)
status_df = status_df[['name_key', 'Status', 'owner_estimate']].drop_duplicates()

# Join everything together
build_out_df = roster_df.merge(
    stats_df[['name_key', 'Team', 'RkOv', 'Age']],
    on='name_key',
    how='left'
)
build_out_df = build_out_df.merge(
    status_df.rename(columns={'owner_estimate': 'Status Owner'}),
    on='name_key',
    how='left'
)

# Ensure requested columns and fallback logic
build_out_df['Team Owner'] = build_out_df['Team Owner'].fillna(build_out_df['Status Owner'])
build_out_df['Status'] = build_out_df['Status'].fillna('Unknown')
build_out_df['Player Contract Years'] = pd.to_numeric(build_out_df['Player Contract Years'], errors='coerce').fillna(1).astype(int)

build_out_df = build_out_df[[
    'Team Owner', 'Player', 'Team', 'Position', 'Player Contract Years', 'RkOv', 'Age', 'Status'
]].copy()

print(f'Built {build_out_df.shape[0]:,} rows')
display(build_out_df.head(20))


Built 180 rows


,Team Owner,Player,Team,Position,Player Contract Years,RkOv,Age,Status
0,Boe,Michael Busch,CHC,1B,4,208.0,28.0,BPP
1,Boe,Kazuma Okamoto,TOR,1B,2,87.0,30.0,BPP
2,Boe,JJ Wetherholt,STL,2B,7,84.0,23.0,BPP
3,Boe,Bo Bischette,NaN,3B,5,NaN,NaN,Unknown
4,Boe,Samuel Basallo,BAL,C,7,354.0,21.0,BPP
5,Boe,Roman Anthony,BOS,OF,7,988.0,22.0,BPP
6,Boe,Wyatt Langford,TEX,OF,7,461.0,24.0,BPP
7,Boe,Addison Barger,TOR,OF,2,1234.0,26.0,FA
8,Boe,Bobby Abreu,NaN,RP,2,NaN,NaN,Unknown
9,Boe,Nolan McLean,NaN,SP,7,NaN,NaN,BPP


# Player Statistics Load

## Hitter Stats

In [21]:
# Load the Fantrax all-hitter_stats export into a DataFrame
fantrax_all_stats_path = source_files['fantrax_all_hitter_stats']

if fantrax_all_stats_path.exists():
    fantrax_all_hitter_stats_df = pd.read_csv(fantrax_all_stats_path)
    print(f'Loaded {fantrax_all_hitter_stats_df.shape[0]:,} rows and {fantrax_all_hitter_stats_df.shape[1]} columns')
    print('Columns:', list(fantrax_all_hitter_stats_df.columns)[:20])
    print('\nPreview:')
    display(fantrax_all_hitter_stats_df.head(10))
    print('\nData types:')
    display(fantrax_all_hitter_stats_df.dtypes)
    print('\nMissing values:')
    display(fantrax_all_hitter_stats_df.isna().sum().sort_values(ascending=False).head(20))
else:
    fantrax_all_stats_df = None
    print(f'Missing file: {fantrax_all_stats_path}')


Loaded 3,352 rows and 19 columns
Columns: ['ID', 'Player', 'Team', 'Position', 'RkOv', 'Status', 'Age', 'Opponent', 'Score', 'Ros', '+/-', 'AB', 'R', 'H', 'HR', 'RBI', 'SB', 'OBP', 'GP']

Preview:


,ID,Player,Team,Position,RkOv,Status,Age,Opponent,Score,Ros,+/-,AB,R,H,HR,RBI,SB,OBP,GP
0,*05rat*,James Wood,WSH,OF,2,DDDS,23,@ATL Sat 7:15PM,100.00,100%,0%,427,99,115,30,72,17,0.398,111
1,*02yc4*,Shohei Ohtani,LAD,"UT,SP",3,Gloggs,32,BOS Sat 9:10PM,99.70,100%,0%,385,70,112,24,67,6,0.394,104
2,*04qk8*,CJ Abrams,WSH,SS,5,AOP,25,@ATL Sat 7:15PM,99.18,100%,0%,400,69,116,28,83,19,0.367,107
3,*051wv*,Pete Crow-Armstrong,CHC,OF,6,IL 10,24,NYY Sat 7:15PM,98.90,100%,0%,414,71,117,24,63,26,0.379,110
4,*05gbf*,Jordan Walker,STL,OF,13,CHAS,24,STL 1<br/>@TOR 5 F,90.50,96%,0%,422,67,119,22,79,14,0.342,109
5,*04355*,Yordan Alvarez,HOU,OF,15,PitlockM,29,TEX Sat 7:10PM,89.94,100%,0%,395,74,128,35,82,1,0.436,109
6,*05u9o*,Sal Stewart,CIN,"1B,3B",19,🍉,22,PIT Sat 6:40PM,87.96,99%,0%,415,58,106,23,84,12,0.334,109
7,*04ru7*,Miguel Vargas,CHW,"1B,3B",20,PitlockM,26,CHW 0<br/>@TB 1,87.44,98%,0%,403,73,96,23,64,11,0.344,109
8,*04b00*,Brice Turang,MIL,2B,21,AOP,26,@LAA Sat 9:38PM,86.94,100%,0%,402,71,109,16,68,15,0.366,104
9,*05yo5*,Ben Rice,NYY,"C,1B",23,AOP,27,@CHC Sat 7:15PM,86.60,100%,0%,394,70,106,31,73,2,0.358,105



Data types:


ID              str
Player          str
Team            str
Position        str
RkOv          int64
Status          str
Age           int64
Opponent        str
Score       float64
Ros             str
+/-             str
AB            int64
R             int64
H             int64
HR            int64
RBI           int64
SB            int64
OBP         float64
GP            int64
dtype: object


Missing values:


Opponent    2
ID          0
+/-         0
OBP         0
SB          0
RBI         0
HR          0
H           0
R           0
AB          0
Ros         0
Player      0
Score       0
Age         0
Status      0
RkOv        0
Position    0
Team        0
GP          0
dtype: int64

In [ ]:
## Pitcher Stats

In [22]:
# Load the Fantrax all-hitter_stats export into a DataFrame
fantrax_all_stats_path = source_files['fantrax_all_pitcher_stats']

if fantrax_all_stats_path.exists():
    fantrax_all_pitcher_stats_df = pd.read_csv(fantrax_all_stats_path)
    print(f'Loaded {fantrax_all_pitcher_stats_df.shape[0]:,} rows and {fantrax_all_pitcher_stats_df.shape[1]} columns')
    print('Columns:', list(fantrax_all_pitcher_stats_df.columns)[:20])
    print('\nPreview:')
    display(fantrax_all_pitcher_stats_df.head(10))
    print('\nData types:')
    display(fantrax_all_pitcher_stats_df.dtypes)
    print('\nMissing values:')
    display(fantrax_all_pitcher_stats_df.isna().sum().sort_values(ascending=False).head(20))
else:
    fantrax_all_pitcher_stats_df = None
    print(f'Missing file: {fantrax_all_stats_path}')


Loaded 3,752 rows and 18 columns
Columns: ['ID', 'Player', 'Team', 'Position', 'RkOv', 'Status', 'Age', 'Opponent', 'Score', 'Ros', '+/-', 'IP', 'ERA', 'WHIP', 'K', 'QS', 'SVH', 'GP']

Preview:


,ID,Player,Team,Position,RkOv,Status,Age,Opponent,Score,Ros,+/-,IP,ERA,WHIP,K,QS,SVH,GP
0,*05u6q*,Cam Schlittler,NYY,SP,1,AOP,25,@CHC Sat 7:15PM,100.00,99%,0%,136.2,2.04,0.907,165,14,0,23
1,*02yc4*,Shohei Ohtani,LAD,"UT,SP",3,Gloggs,32,BOS Sat 9:10PM,99.70,100%,0%,85.2,1.79,0.946,95,12,0,104
2,*061yu*,Jacob Misiorowski,MIL,SP,4,BPP,24,@LAA Sat 9:38PM,99.44,100%,0%,120.0,1.57,0.750,185,12,0,20
3,*01uca*,Chris Sale,ATL,SP,7,Gloggs,37,WSH Sat 7:15PM,96.03,100%,0%,117.0,2.08,1.017,143,13,0,20
4,*05r8n*,Yoshinobu Yamamoto,LAD,SP,8,AOP,27,BOS Sat 9:10PM,95.12,100%,0%,125.2,2.72,0.899,118,15,0,19
5,*05ucd*,Mason Miller,SD,RP,9,Gloggs,27,SF Sat 8:40PM,94.50,99%,0%,45.2,0.79,0.788,85,0,28,44
6,*052c8*,Cristopher Sanchez,PHI,SP,10,CHAS,29,@BAL Sat 7:05PM,92.05,100%,0%,138.2,2.73,1.204,157,15,0,22
7,*01sb3*,Zack Wheeler,PHI,SP,11,Gloggs,36,@BAL Sat 7:05PM,91.79,99%,0%,103.0,2.53,0.922,123,12,0,17
8,*051bm*,Louis Varland,TOR,RP,12,DDDS,28,STL 1<br/>@TOR 5 F,91.50,89%,0%,56.0,0.96,0.964,73,0,29,50
9,*03df2*,Dylan Cease,TOR,SP,14,CHAS,30,STL 1<br/>@TOR 5 F,90.22,100%,0%,119.1,2.41,1.081,174,10,0,20



Data types:


ID              str
Player          str
Team            str
Position        str
RkOv          int64
Status          str
Age           int64
Opponent        str
Score       float64
Ros             str
+/-             str
IP          float64
ERA         float64
WHIP        float64
K             int64
QS            int64
SVH           int64
GP            int64
dtype: object


Missing values:


Opponent    1
ID          0
+/-         0
SVH         0
QS          0
K           0
WHIP        0
ERA         0
IP          0
Ros         0
Player      0
Score       0
Age         0
Status      0
RkOv        0
Position    0
Team        0
GP          0
dtype: int64

# Player Ranking Loads

## Harry's Ranking

In [23]:
# Load the harryknowsball ranking export into a DataFrame
harryknowsball_path = source_files['harryknowsball']

if harryknowsball_path.exists():
    harry_rankings_df = pd.read_csv(harryknowsball_path)
    print(f'Loaded {harry_rankings_df.shape[0]:,} rows and {harry_rankings_df.shape[1]} columns')
    print('Columns:', list(harry_rankings_df.columns)[:20])
    print('\nPreview:')
    display(harry_rankings_df.head(10))
else:
    harry_rankings_df = None
    print(f'Missing file: {harryknowsball_path}')

Loaded 1,746 rows and 7 columns
Columns: ['Rank', 'Name', 'Value', 'Age', 'Positions', 'Team', 'Level']

Preview:


,Rank,Name,Value,Age,Positions,Team,Level
0,1,Shohei Ohtani,10000,32.1,"SP, UT",LAD,MLB
1,2,Bobby Witt Jr.,9911,26.1,SS,KC,MLB
2,3,Nick Kurtz,7860,23.4,1B,ATH,MLB
3,4,James Wood,7667,23.9,OF,WSH,MLB
4,5,Junior Caminero,7645,23.1,3B,TB,MLB
5,6,Jacob Misiorowski,7293,24.3,SP,MIL,MLB
6,7,Juan Soto,7112,27.8,OF,NYM,MLB
7,8,Paul Skenes,7104,24.2,SP,PIT,MLB
8,9,Elly De La Cruz,6920,24.5,SS,CIN,MLB
9,10,Konnor Griffin,6473,20.3,SS,PIT,MLB


## Keiths Ranking

In [25]:
# Load the Keith prospects export into a DataFrame
keith_prospects_path = source_files['keith_prospects']

if keith_prospects_path.exists():
    keith_prospects_df = pd.read_csv(keith_prospects_path)
    print(f'Loaded {keith_prospects_df.shape[0]:,} rows and {keith_prospects_df.shape[1]} columns')
    print('Columns:', list(keith_prospects_df.columns)[:20])
    print('\nPreview:')
    html_table = keith_prospects_df.head(100).to_html(index=False)
    scrollable_html = f"<div style='max-height:500px; overflow:auto; border:1px solid #ddd; padding:8px;'>{html_table}</div>"
    display(HTML(scrollable_html))
else:
    keith_prospects_df = None
    print(f'Missing file: {keith_prospects_path}')


Loaded 1,396 rows and 5 columns
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

Preview:


Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
NaN,1,NaN,NaN,NaN
NaN,Kade Anderson,NaN,NaN,NaN
NaN,LHP,NaN,NaN,NaN
NaN,Seattle Mariners,NaN,NaN,NaN
NaN,Age:,NaN,NaN,NaN
NaN,22,NaN,NaN,NaN
NaN,Ht:,NaN,NaN,NaN
NaN,6-2,NaN,NaN,NaN
NaN,Wt:,NaN,NaN,NaN
NaN,179,NaN,NaN,NaN


In [28]:
# Reshape the ranking export from 18-row blocks into one record per block
if 'keith_prospects_df' in globals() and keith_prospects_df is not None and not keith_prospects_df.empty:
    ranking_source = keith_prospects_df.iloc[:, 1].dropna().astype(str).reset_index(drop=True)
    block_size = 20
    records = []
    for start in range(0, len(ranking_source), block_size):
        block = ranking_source.iloc[start:start + block_size].tolist()
        if len(block) < block_size:
            continue
        records.append(block[:6])

    ranking_list_df = pd.DataFrame(records)
    ranking_list_df.columns = [f'rank_{i+1}' for i in range(ranking_list_df.shape[1])]
    print(f'Created {ranking_list_df.shape[0]:,} ranking records from {len(ranking_source):,} values')
    display(ranking_list_df.head(10))
else:
    ranking_list_df = pd.DataFrame()
    print('No ranking data available to reshape')


Created 56 ranking records from 1,122 values


,rank_1,rank_2,rank_3,rank_4,rank_5,rank_6
0,1,Kade Anderson,LHP,Seattle Mariners,Age:,22
1,Age:,19,Ht:,6-1,Wt:,187
2,Wt:,183,Bats:,B,Throws:,R
3,Throws:,R,Boston Red Sox,SS,5,Theo Gillen
4,6,George Lombard Jr.,SS,New York Yankees,Age:,21
5,Age:,20,Ht:,6-4,Wt:,190
6,6-5,Wt:,220,Bats:,R,Throws:
7,R,Throws:,R,Pittsburgh Pirates,RHP,10
8,SS,11,Ethan Salas,C,San Diego Padres,Age:
9,San Francisco Giants,Age:,18,Ht:,6-0,Wt:
